In [0]:
USE CATALOG meridian_t02;

-- 1. Engagement and Booking Conversion Metric View
CREATE OR REPLACE VIEW gold.mv_stays_kpis
WITH METRICS LANGUAGE YAML AS $$
version: 1.1
source: meridian_t02.gold.fct_sessions

fields:
  - name: user_country
    expr: user_country
    display_name: 'User Country'
    comment: 'Country of the user session; dimension for regional row filters'

  - name: device_type
    expr: device_type
    display_name: 'Device Type'
    comment: 'Client device extracted from clickstream metadata'

  - name: user_type
    expr: user_type
    display_name: 'User Type'
    comment: 'User tier at session time from SCD Type 2 history'

  - name: session_date
    expr: CAST(session_start_time AS DATE)
    display_name: 'Session Date'
    comment: 'Date the browsing session commenced'

measures:
  - name: active_users
    expr: COUNT(DISTINCT user_id)
    display_name: 'Active Users'
    comment: 'Count of unique active users browsing the platform'

  - name: total_sessions
    expr: COUNT(DISTINCT session_id)
    display_name: 'Total Sessions'
    comment: 'Total unique sessions identified after 30-min inactivity breaks'

  - name: events_per_session
    expr: SUM(event_count) / COUNT(DISTINCT session_id)
    display_name: 'Events Per Session'
    comment: 'Interaction intensity per session (Ratio of sums)'

  - name: search_to_booking_conversion
    expr: COUNT(DISTINCT CASE WHEN is_booked = 1 THEN session_id END) / COUNT(DISTINCT session_id)
    display_name: 'Booking Conversion Rate'
    comment: 'Share of sessions that concluded in a booking (Ratio of sums)'
$$;

-- 2. Reviews Sentiment and Soft-Deletion Metric View
CREATE OR REPLACE VIEW gold.mv_reviews_kpis
WITH METRICS LANGUAGE YAML AS $$
version: 1.1
source: meridian_t02.gold.fct_reviews

fields:
  - name: user_country
    expr: user_country
    display_name: 'User Country'
    comment: 'Country of the reviewing user'

  - name: review_date
    expr: CAST(review_timestamp AS DATE)
    display_name: 'Review Date'
    comment: 'Submission date of the review'

measures:
  - name: total_reviews
    expr: COUNT(review_id)
    display_name: 'Total Reviews'
    comment: 'Total count of submitted reviews'

  - name: average_rating
    expr: AVG(rating)
    display_name: 'Average Rating'
    comment: 'Mean property rating across valid reviews'

  - name: deleted_review_share
    expr: SUM(CASE WHEN is_deleted THEN 1 ELSE 0 END) / COUNT(*)
    display_name: 'Deleted Review Share'
    comment: 'Share of reviews marked as soft-deleted by moderation (Ratio of sums)'
$$;